In [1]:
import pandas as pd
import re

# Carica il CSV
df = pd.read_csv("codice_penale.csv")

def estrai_numero_articolo(val):
    """
    Estrae la parte numerica principale dell'articolo.
    Esempi:
    '240-bis' -> 240
    '734 bis' -> 734
    """
    if pd.isna(val):
        return None
    match = re.search(r'\d+', str(val))
    return int(match.group()) if match else None

def assegna_libro(articolo):
    if articolo is None:
        return None
    if 1 <= articolo <= 240:
        return "Libro I"
    elif 241 <= articolo <= 649:
        return "Libro II"
    elif 650 <= articolo <= 734:
        return "Libro III"
    else:
        return "Fuori range"

# Supponendo che la colonna si chiami 'articolo'
df["articolo_num"] = df["articolo"].apply(estrai_numero_articolo)
df["libro_codice_penale"] = df["articolo_num"].apply(assegna_libro)

# (Opzionale) rimuove la colonna di supporto
df.drop(columns=["articolo_num"], inplace=True)

# Salva il nuovo CSV
df.to_csv("codice_penale_con_libro.csv", index=False)


In [3]:
import pandas as pd
import re

# Carica il CSV
df = pd.read_csv("codice_civile.csv")

def estrai_numero_articolo(article_id):
    """
    Estrae il numero dell'articolo da valori tipo:
    'art1' -> 1
    'art840sexiesdecies' -> 840
    'art633bis' -> 633
    """
    if pd.isna(article_id):
        return None
    match = re.match(r'art(\d+)', str(article_id))
    return int(match.group(1)) if match else None

def assegna_libro_civile(num):
    if num is None:
        return None
    if 1 <= num <= 162:
        return "Libro Primo"
    elif 163 <= num <= 473:
        return "Libro Secondo"
    elif 474 <= num <= 632:
        return "Libro Terzo"
    elif 633 <= num <= 840:
        return "Libro Quarto"
    else:
        return "Fuori range"

# Estrazione numero articolo
df["articolo_num"] = df["article_id"].apply(estrai_numero_articolo)

# Assegnazione libro
df["libro_codice_civile"] = df["articolo_num"].apply(assegna_libro_civile)

# Rimuove la colonna tecnica
df.drop(columns=["articolo_num"], inplace=True)

# Salvataggio
df.to_csv("codice_civile_con_libro.csv", index=False)


In [4]:
pip install pandas torch transformers tqdm


In [7]:
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

# =========================
# Config
# =========================
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 512

FILES = {
    "civile": {
        "path": "codice_civile_con_libro.csv",
        "title_col": "article_title",
        "text_col": "article_text"
    },
    "penale": {
        "path": "codice_penale_con_libro.csv",
        "title_col": "titolo",
        "text_col": "testo"
    }
}

# =========================
# Load model
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# =========================
# Mean pooling
# =========================
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * mask).sum(1) / mask.sum(1)

# =========================
# Embedding
# =========================
def embed_texts(texts):
    embeddings = []

    for text in tqdm(texts):
        inputs = tokenizer(
            text,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        ).to(DEVICE)

        with torch.no_grad():
            outputs = model(**inputs)

        emb = mean_pooling(outputs, inputs["attention_mask"])
        embeddings.append(emb.cpu().numpy()[0])

    return np.vstack(embeddings)

# =========================
# Process files
# =========================
for label, cfg in FILES.items():
    print(f"\n▶ Embedding {label}")

    df = pd.read_csv(cfg["path"])

    # 🔑 testo effettivo da embeddare
    texts = (
        df[cfg["title_col"]].fillna("").astype(str) + "\n" +
        df[cfg["text_col"]].fillna("").astype(str)
    ).tolist()

    embeddings = embed_texts(texts)

    # Salvataggio
    np.save(f"{label}_embeddings.npy", embeddings)
    df.to_pickle(f"{label}_metadata.pkl")

    print(f"✔ Salvati {label}_embeddings.npy e {label}_metadata.pkl")



▶ Embedding civile


100%|██████████| 3039/3039 [00:38<00:00, 79.59it/s]


✔ Salvati civile_embeddings.npy e civile_metadata.pkl

▶ Embedding penale


100%|██████████| 925/925 [00:16<00:00, 56.19it/s]

✔ Salvati penale_embeddings.npy e penale_metadata.pkl
